# 06 — Validate Gold

## Objective

Validate that the Gold analytical model was correctly created and is ready for reporting.

## This notebook performs

- Validation of Gold table existence.
- Validation of row counts.
- Validation of surrogate keys.
- Validation of foreign key relationships between facts and dimensions.
- Validation of business metric ranges.
- Creation of a persistent Gold quality log table.

## Gold tables validated

Dimensions:

- `dim_region`
- `dim_date`
- `dim_employee`
- `dim_client`

Facts:

- `fact_workforce_monthly`
- `fact_goals_monthly`

## Notes

This notebook does not transform data. It audits the output of `05_build_gold` and stores validation results as evidence of Gold layer quality.

In [0]:
# Import required libraries for validation logging.

from datetime import datetime
from pyspark.sql import functions as F

# Use the project schema where Gold tables were created.

spark.sql("USE SCHEMA workbridge")

DataFrame[]

In [0]:
# Read Gold dimensions and fact tables created in 05_build_gold.

dim_region_df = spark.table("dim_region")
dim_date_df = spark.table("dim_date")
dim_employee_df = spark.table("dim_employee")
dim_client_df = spark.table("dim_client")

fact_workforce_monthly_df = spark.table("fact_workforce_monthly")
fact_goals_monthly_df = spark.table("fact_goals_monthly")

In [0]:
# This list will store the result of each Gold validation check.

gold_validation_results = []

# Helper function to register validation results in a consistent structure.

def add_gold_validation_result(table_name, check_name, status, records_count=None, message=""):
    """
    Add a Gold validation result to the gold_validation_results list.

    Parameters:
    - table_name: name of the table being validated.
    - check_name: name of the validation check.
    - status: PASS or FAIL.
    - records_count: number of records involved in the check.
    - message: short explanation of the result.
    """

    gold_validation_results.append({
        "validation_timestamp": datetime.now(),
        "table_name": table_name,
        "check_name": check_name,
        "status": status,
        "records_count": records_count,
        "message": message,
    })

# General Gold Table Checks

## Objective

Validate that all Gold tables exist and contain records.

In [0]:
# Validate that all Gold tables exist and contain records.

gold_tables = [
    "dim_region",
    "dim_date",
    "dim_employee",
    "dim_client",
    "fact_workforce_monthly",
    "fact_goals_monthly",
]

for table_name in gold_tables:
    try:
        df = spark.table(table_name)
        row_count = df.count()

        # Register table existence.
        add_gold_validation_result(
            table_name=table_name,
            check_name="table_exists_check",
            status="PASS",
            records_count=row_count,
            message="Table exists."
        )

        # Register whether the table contains records.
        add_gold_validation_result(
            table_name=table_name,
            check_name="row_count_check",
            status="PASS" if row_count > 0 else "FAIL",
            records_count=row_count,
            message="Table contains records." if row_count > 0 else "Table is empty."
        )

    except Exception as error:
        # Register failure if the table cannot be read.
        add_gold_validation_result(
            table_name=table_name,
            check_name="table_exists_check",
            status="FAIL",
            records_count=None,
            message=f"Table could not be read: {str(error)}"
        )

# Dimension Key Checks

## Objective

Validate that each Gold dimension has a non-null and unique surrogate key.

In [0]:
# Validate dim_region surrogate key and business code.

region_key_null_count = dim_region_df.filter(F.col("region_key").isNull()).count()

add_gold_validation_result(
    table_name="dim_region",
    check_name="region_key_not_null_check",
    status="PASS" if region_key_null_count == 0 else "FAIL",
    records_count=region_key_null_count,
    message="region_key is not null." if region_key_null_count == 0 else "region_key contains null values."
)

region_key_duplicate_count = (
    dim_region_df
    .groupBy("region_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

add_gold_validation_result(
    table_name="dim_region",
    check_name="region_key_unique_check",
    status="PASS" if region_key_duplicate_count == 0 else "FAIL",
    records_count=region_key_duplicate_count,
    message="region_key is unique." if region_key_duplicate_count == 0 else "Duplicated region_key values found."
)

region_id_duplicate_count = (
    dim_region_df
    .groupBy("region_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

add_gold_validation_result(
    table_name="dim_region",
    check_name="region_id_unique_check",
    status="PASS" if region_id_duplicate_count == 0 else "FAIL",
    records_count=region_id_duplicate_count,
    message="region_id is unique." if region_id_duplicate_count == 0 else "Duplicated region_id values found."
)

In [0]:
# Validate dim_date surrogate key and period.

date_key_null_count = dim_date_df.filter(F.col("date_key").isNull()).count()

add_gold_validation_result(
    table_name="dim_date",
    check_name="date_key_not_null_check",
    status="PASS" if date_key_null_count == 0 else "FAIL",
    records_count=date_key_null_count,
    message="date_key is not null." if date_key_null_count == 0 else "date_key contains null values."
)

date_key_duplicate_count = (
    dim_date_df
    .groupBy("date_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

add_gold_validation_result(
    table_name="dim_date",
    check_name="date_key_unique_check",
    status="PASS" if date_key_duplicate_count == 0 else "FAIL",
    records_count=date_key_duplicate_count,
    message="date_key is unique." if date_key_duplicate_count == 0 else "Duplicated date_key values found."
)

period_duplicate_count = (
    dim_date_df
    .groupBy("period")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

add_gold_validation_result(
    table_name="dim_date",
    check_name="period_unique_check",
    status="PASS" if period_duplicate_count == 0 else "FAIL",
    records_count=period_duplicate_count,
    message="period is unique." if period_duplicate_count == 0 else "Duplicated period values found."
)

In [0]:
# Validate dim_employee surrogate key, original employee ID and region relationship.

# Check that employee_key is not null.
employee_key_null_count = (
    dim_employee_df
    .filter(F.col("employee_key").isNull())
    .count()
)

add_gold_validation_result(
    table_name="dim_employee",
    check_name="employee_key_not_null_check",
    status="PASS" if employee_key_null_count == 0 else "FAIL",
    records_count=employee_key_null_count,
    message="employee_key is not null."
    if employee_key_null_count == 0 else "employee_key contains null values."
)

# Check that employee_key is unique.
employee_key_duplicate_count = (
    dim_employee_df
    .groupBy("employee_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

add_gold_validation_result(
    table_name="dim_employee",
    check_name="employee_key_unique_check",
    status="PASS" if employee_key_duplicate_count == 0 else "FAIL",
    records_count=employee_key_duplicate_count,
    message="employee_key is unique."
    if employee_key_duplicate_count == 0 else "Duplicated employee_key values found."
)

# Check that employee_id is unique in the current initial-load dimension.
# This is expected because we do not have historical SCD versions yet.
employee_id_duplicate_count = (
    dim_employee_df
    .groupBy("employee_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

add_gold_validation_result(
    table_name="dim_employee",
    check_name="employee_id_unique_check",
    status="PASS" if employee_id_duplicate_count == 0 else "FAIL",
    records_count=employee_id_duplicate_count,
    message="employee_id is unique for the current initial-load dimension."
    if employee_id_duplicate_count == 0 else "Duplicated employee_id values found."
)

# Monitor employees without region_key.
# This is warning-level because employee region was not treated as critical in Silver.
employee_region_key_null_count = (
    dim_employee_df
    .filter(F.col("region_key").isNull())
    .count()
)

add_gold_validation_result(
    table_name="dim_employee",
    check_name="employee_region_key_null_warning_check",
    status="PASS",
    records_count=employee_region_key_null_count,
    message=f"{employee_region_key_null_count} employees have null region_key and should be reviewed."
)

In [0]:
# Validate dim_client surrogate key, original client ID and region relationship.

# Check that client_key is not null.
client_key_null_count = (
    dim_client_df
    .filter(F.col("client_key").isNull())
    .count()
)

add_gold_validation_result(
    table_name="dim_client",
    check_name="client_key_not_null_check",
    status="PASS" if client_key_null_count == 0 else "FAIL",
    records_count=client_key_null_count,
    message="client_key is not null."
    if client_key_null_count == 0 else "client_key contains null values."
)

# Check that client_key is unique.
client_key_duplicate_count = (
    dim_client_df
    .groupBy("client_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

add_gold_validation_result(
    table_name="dim_client",
    check_name="client_key_unique_check",
    status="PASS" if client_key_duplicate_count == 0 else "FAIL",
    records_count=client_key_duplicate_count,
    message="client_key is unique."
    if client_key_duplicate_count == 0 else "Duplicated client_key values found."
)

# Check that client_id is unique in the Gold dimension.
client_id_duplicate_count = (
    dim_client_df
    .groupBy("client_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

add_gold_validation_result(
    table_name="dim_client",
    check_name="client_id_unique_check",
    status="PASS" if client_id_duplicate_count == 0 else "FAIL",
    records_count=client_id_duplicate_count,
    message="client_id is unique."
    if client_id_duplicate_count == 0 else "Duplicated client_id values found."
)

# Check that all clients are linked to a valid region.
# Client region was treated as critical in Silver, so region_key should not be null in Gold.
client_region_key_null_count = (
    dim_client_df
    .filter(F.col("region_key").isNull())
    .count()
)

add_gold_validation_result(
    table_name="dim_client",
    check_name="client_region_key_not_null_check",
    status="PASS" if client_region_key_null_count == 0 else "FAIL",
    records_count=client_region_key_null_count,
    message="All clients have a valid region_key."
    if client_region_key_null_count == 0 else "Some clients have null region_key."
)

# Fact Key Checks

## Objective

Validate that each Gold fact table has a non-null and unique surrogate key.

In [0]:
# Validate fact_workforce_monthly surrogate key.

workforce_fact_key_null_count = (
    fact_workforce_monthly_df
    .filter(F.col("workforce_fact_key").isNull())
    .count()
)

add_gold_validation_result(
    table_name="fact_workforce_monthly",
    check_name="workforce_fact_key_not_null_check",
    status="PASS" if workforce_fact_key_null_count == 0 else "FAIL",
    records_count=workforce_fact_key_null_count,
    message="workforce_fact_key is not null."
    if workforce_fact_key_null_count == 0 else "workforce_fact_key contains null values."
)

workforce_fact_key_duplicate_count = (
    fact_workforce_monthly_df
    .groupBy("workforce_fact_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

add_gold_validation_result(
    table_name="fact_workforce_monthly",
    check_name="workforce_fact_key_unique_check",
    status="PASS" if workforce_fact_key_duplicate_count == 0 else "FAIL",
    records_count=workforce_fact_key_duplicate_count,
    message="workforce_fact_key is unique."
    if workforce_fact_key_duplicate_count == 0 else "Duplicated workforce_fact_key values found."
)

In [0]:
# Validate fact_goals_monthly surrogate key.

goals_fact_key_null_count = (
    fact_goals_monthly_df
    .filter(F.col("goals_fact_key").isNull())
    .count()
)

add_gold_validation_result(
    table_name="fact_goals_monthly",
    check_name="goals_fact_key_not_null_check",
    status="PASS" if goals_fact_key_null_count == 0 else "FAIL",
    records_count=goals_fact_key_null_count,
    message="goals_fact_key is not null."
    if goals_fact_key_null_count == 0 else "goals_fact_key contains null values."
)

goals_fact_key_duplicate_count = (
    fact_goals_monthly_df
    .groupBy("goals_fact_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

add_gold_validation_result(
    table_name="fact_goals_monthly",
    check_name="goals_fact_key_unique_check",
    status="PASS" if goals_fact_key_duplicate_count == 0 else "FAIL",
    records_count=goals_fact_key_duplicate_count,
    message="goals_fact_key is unique."
    if goals_fact_key_duplicate_count == 0 else "Duplicated goals_fact_key values found."
)

# Referential Integrity Checks

## Objective

Validate that fact table foreign keys correctly reference existing Gold dimension keys.

In [0]:
# Prepare valid dimension keys.

valid_date_keys_df = dim_date_df.select("date_key").distinct()
valid_employee_keys_df = dim_employee_df.select("employee_key").distinct()
valid_client_keys_df = dim_client_df.select("client_key").distinct()
valid_region_keys_df = dim_region_df.select("region_key").distinct()

In [0]:
# Validate fact_workforce_monthly foreign keys.

invalid_workforce_date_fk_count = (
    fact_workforce_monthly_df
    .join(valid_date_keys_df, on="date_key", how="left_anti")
    .count()
)

add_gold_validation_result(
    table_name="fact_workforce_monthly",
    check_name="date_fk_check",
    status="PASS" if invalid_workforce_date_fk_count == 0 else "FAIL",
    records_count=invalid_workforce_date_fk_count,
    message="All workforce date_key values exist in dim_date."
    if invalid_workforce_date_fk_count == 0 else "Some workforce date_key values do not exist in dim_date."
)

invalid_workforce_employee_fk_count = (
    fact_workforce_monthly_df
    .join(valid_employee_keys_df, on="employee_key", how="left_anti")
    .count()
)

add_gold_validation_result(
    table_name="fact_workforce_monthly",
    check_name="employee_fk_check",
    status="PASS" if invalid_workforce_employee_fk_count == 0 else "FAIL",
    records_count=invalid_workforce_employee_fk_count,
    message="All workforce employee_key values exist in dim_employee."
    if invalid_workforce_employee_fk_count == 0 else "Some workforce employee_key values do not exist in dim_employee."
)

invalid_workforce_client_fk_count = (
    fact_workforce_monthly_df
    .join(valid_client_keys_df, on="client_key", how="left_anti")
    .count()
)

add_gold_validation_result(
    table_name="fact_workforce_monthly",
    check_name="client_fk_check",
    status="PASS" if invalid_workforce_client_fk_count == 0 else "FAIL",
    records_count=invalid_workforce_client_fk_count,
    message="All workforce client_key values exist in dim_client."
    if invalid_workforce_client_fk_count == 0 else "Some workforce client_key values do not exist in dim_client."
)

invalid_workforce_region_fk_count = (
    fact_workforce_monthly_df
    .join(valid_region_keys_df, on="region_key", how="left_anti")
    .count()
)

add_gold_validation_result(
    table_name="fact_workforce_monthly",
    check_name="region_fk_check",
    status="PASS" if invalid_workforce_region_fk_count == 0 else "FAIL",
    records_count=invalid_workforce_region_fk_count,
    message="All workforce region_key values exist in dim_region."
    if invalid_workforce_region_fk_count == 0 else "Some workforce region_key values do not exist in dim_region."
)

In [0]:
# Validate fact_goals_monthly foreign keys.

invalid_goals_date_fk_count = (
    fact_goals_monthly_df
    .join(valid_date_keys_df, on="date_key", how="left_anti")
    .count()
)

add_gold_validation_result(
    table_name="fact_goals_monthly",
    check_name="date_fk_check",
    status="PASS" if invalid_goals_date_fk_count == 0 else "FAIL",
    records_count=invalid_goals_date_fk_count,
    message="All goals date_key values exist in dim_date."
    if invalid_goals_date_fk_count == 0 else "Some goals date_key values do not exist in dim_date."
)

invalid_goals_client_fk_count = (
    fact_goals_monthly_df
    .join(valid_client_keys_df, on="client_key", how="left_anti")
    .count()
)

add_gold_validation_result(
    table_name="fact_goals_monthly",
    check_name="client_fk_check",
    status="PASS" if invalid_goals_client_fk_count == 0 else "FAIL",
    records_count=invalid_goals_client_fk_count,
    message="All goals client_key values exist in dim_client."
    if invalid_goals_client_fk_count == 0 else "Some goals client_key values do not exist in dim_client."
)

invalid_goals_region_fk_count = (
    fact_goals_monthly_df
    .join(valid_region_keys_df, on="region_key", how="left_anti")
    .count()
)

add_gold_validation_result(
    table_name="fact_goals_monthly",
    check_name="region_fk_check",
    status="PASS" if invalid_goals_region_fk_count == 0 else "FAIL",
    records_count=invalid_goals_region_fk_count,
    message="All goals region_key values exist in dim_region."
    if invalid_goals_region_fk_count == 0 else "Some goals region_key values do not exist in dim_region."
)

# Fact Metric Checks

## Objective

Validate that Gold fact metrics are within expected business ranges.

In [0]:
# Validate workforce fact metrics.
# Utilization can be greater than 1 when overtime exists, so the upper threshold is set to 2.0.

invalid_workforce_metrics_count = (
    fact_workforce_monthly_df
    .filter(
        (F.col("available_hours").isNull()) | (F.col("available_hours") <= 0)
        | (F.col("worked_hours").isNull()) | (F.col("worked_hours") < 0)
        | (F.col("billable_hours").isNull()) | (F.col("billable_hours") < 0)
        | (F.col("billable_hours") > F.col("worked_hours"))
        | (F.col("non_billable_hours").isNull()) | (F.col("non_billable_hours") < 0)
        | (F.col("overtime_hours").isNull()) | (F.col("overtime_hours") < 0)
        | (F.col("utilization_rate").isNull()) | (F.col("utilization_rate") < 0) | (F.col("utilization_rate") > 2.0)
        | (F.col("billable_rate").isNull()) | (F.col("billable_rate") < 0) | (F.col("billable_rate") > 1)
    )
    .count()
)

add_gold_validation_result(
    table_name="fact_workforce_monthly",
    check_name="workforce_metrics_validity_check",
    status="PASS" if invalid_workforce_metrics_count == 0 else "FAIL",
    records_count=invalid_workforce_metrics_count,
    message="All workforce metrics are within expected ranges."
    if invalid_workforce_metrics_count == 0 else "Invalid workforce metric values found."
)

In [0]:
# Validate workforce cost metrics.
# Some cost values may be null if no matching employee-month cost record exists,
# but when present they should not be negative.

invalid_workforce_cost_metrics_count = (
    fact_workforce_monthly_df
    .filter(
        (F.col("salary_cost").isNotNull() & (F.col("salary_cost") < 0))
        | (F.col("benefits_cost").isNotNull() & (F.col("benefits_cost") < 0))
        | (F.col("total_cost").isNotNull() & (F.col("total_cost") < 0))
    )
    .count()
)

add_gold_validation_result(
    table_name="fact_workforce_monthly",
    check_name="workforce_cost_metrics_validity_check",
    status="PASS" if invalid_workforce_cost_metrics_count == 0 else "FAIL",
    records_count=invalid_workforce_cost_metrics_count,
    message="All workforce cost metrics are valid when present."
    if invalid_workforce_cost_metrics_count == 0 else "Invalid workforce cost metric values found."
)

In [0]:
# Validate goals fact metrics.
# Target rates must be between 0 and 1.
# Target cost and target billable hours must be non-negative.

invalid_goals_metrics_count = (
    fact_goals_monthly_df
    .filter(
        (F.col("target_turnover_rate").isNull()) | (F.col("target_turnover_rate") < 0) | (F.col("target_turnover_rate") > 1)
        | (F.col("target_utilization_rate").isNull()) | (F.col("target_utilization_rate") < 0) | (F.col("target_utilization_rate") > 1)
        | (F.col("target_cost").isNull()) | (F.col("target_cost") < 0)
        | (F.col("target_billable_hours").isNull()) | (F.col("target_billable_hours") < 0)
    )
    .count()
)

add_gold_validation_result(
    table_name="fact_goals_monthly",
    check_name="goals_metrics_validity_check",
    status="PASS" if invalid_goals_metrics_count == 0 else "FAIL",
    records_count=invalid_goals_metrics_count,
    message="All goals metrics are within expected ranges."
    if invalid_goals_metrics_count == 0 else "Invalid goals metric values found."
)

# Create Gold Quality Log

## Objective

Create a persistent quality log with all Gold validation results.

In [0]:
# Convert validation results into a Spark DataFrame.

gold_quality_log_df = spark.createDataFrame(gold_validation_results)

# Display the detailed validation log.
display(gold_quality_log_df)

check_name,message,records_count,status,table_name,validation_timestamp
table_exists_check,Table exists.,4,PASS,dim_region,2026-05-09T15:08:59.382Z
row_count_check,Table contains records.,4,PASS,dim_region,2026-05-09T15:08:59.382Z
table_exists_check,Table exists.,12,PASS,dim_date,2026-05-09T15:08:59.676Z
row_count_check,Table contains records.,12,PASS,dim_date,2026-05-09T15:08:59.676Z
table_exists_check,Table exists.,493,PASS,dim_employee,2026-05-09T15:08:59.990Z
row_count_check,Table contains records.,493,PASS,dim_employee,2026-05-09T15:08:59.991Z
table_exists_check,Table exists.,13,PASS,dim_client,2026-05-09T15:09:00.294Z
row_count_check,Table contains records.,13,PASS,dim_client,2026-05-09T15:09:00.294Z
table_exists_check,Table exists.,3292,PASS,fact_workforce_monthly,2026-05-09T15:09:00.598Z
row_count_check,Table contains records.,3292,PASS,fact_workforce_monthly,2026-05-09T15:09:00.598Z


In [0]:
# Save Gold validation results as a Delta table.

gold_quality_log_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("gold_quality_log")

# Gold Validation Summary

## Objective

Show the final count of passed and failed Gold validation checks.

In [0]:
# Show validation summary by status.

gold_validation_summary_df = (
    gold_quality_log_df
    .groupBy("status")
    .agg(F.count("*").alias("checks_count"))
)

display(gold_validation_summary_df)

status,checks_count
PASS,40
